# Financial-Only Models — Predicting Daily Change in Polymarket Trump Probability

Four regression models trained exclusively on financial and macroeconomic features:
1. **Ridge Regression** — regularised linear baseline; alpha tuned by CV  
   *(Plain OLS is severely underdetermined: the first CV fold has only ~5 training rows for 27 features. Ridge is the only valid linear approach here.)*
2. **Random Forest** — tree ensemble; tuned `max_depth`, `min_samples_leaf`
3. **SVM** — kernel regression; tuned `kernel`, `C`, `epsilon`
4. **XGBoost** — gradient boosting; tuned `max_depth`, `learning_rate`, `n_estimators`

**Target:** `Δ polymarket_trump_prob = prob(t) − prob(t−1)` — daily change in Trump win probability  
**Features (27):** S&P 500 (price, returns, rolling stats, lags), oil (price, returns, rolling stats, lags), VIX, 10-year bond yield, USD index, and monthly macro indicators.

> **Critical data limitation.** The monthly macro indicators (`macro_real_income`, `macro_real_gdp`, `macro_unemployment`, `macro_cpi`, `macro_consumer_sentiment`) are only updated monthly/quarterly. After `dropna()` to align all 27 features, the dataset shrinks to ~32 rows covering only Oct–Nov 2024. This means:
> - Training folds have only 5–13 rows
> - All model results carry very high uncertainty
> - CV estimates from 4-row validation folds are noisy proxies at best

> **Note on contemporaneous market features.** Same-day close prices and returns (`sp500_close`, `sp500_return_1d`, `vix_close`, etc.) are end-of-day values — the same timing as the polymarket closing price. Lagged returns (`sp500_return_lag1/2/3`, `oil_return_lag1/2`) are strictly backward-looking.

**Hyperparameter tuning:** simplified grids appropriate for the tiny dataset.  
**Splits:** walk-forward expanding-window CV, 3 folds, gap = 1 day, held-out test = last 14 days  
**Metrics:** MAE, RMSE, Directional Accuracy, R²

## 1. Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError("XGBoost not found. Install with: pip install xgboost")

from Functions.data_splits import (
    get_cv_folds, get_test_split, print_fold_summary, validate_no_leakage,
)
from Functions.evaluation_metrics import (
    directional_accuracy, compute_metrics, cv_evaluate, final_eval, tune_hyperparams,
)
from house_style import (
    apply_style, BG_DARK, BG_PANEL, TEXT_PRIMARY, TEXT_MUTED,
    SPINE_COLOR, GRID_COLOR, DEMOCRAT, REPUBLICAN, NEUTRAL, PALETTE,
)
apply_style()

RANDOM_STATE = 42
TEST_DAYS    = 14
N_SPLITS     = 3
GAP          = 1

DATA_PATH = "../../../Data/3_Gold/basetable.csv"

FIN_COLS = [
    # ── S&P 500 ───────────────────────────────────────────────────────────────────────────
    "sp500_close", "sp500_return_1d", "sp500_return_3d", "sp500_return_7d",
    "sp500_rolling_mean_7d", "sp500_vol_7d", "sp500_range_7d",
    "sp500_return_accel", "sp500_vol_change_7d",
    "sp500_return_lag1", "sp500_return_lag2", "sp500_return_lag3",
    # ── Oil ──────────────────────────────────────────────────────────────────────────────
    "oil_close", "oil_return_1d", "oil_return_7d",
    "oil_rolling_mean_7d", "oil_vol_7d",
    "oil_return_lag1", "oil_return_lag2",
    # ── Volatility / rates / FX ────────────────────────────────────────────────────────
    "vix_close", "bond_10y", "usd_index",
    # ── Macro indicators (monthly/quarterly → severe NaN sparsity) ───────────────────────────
    "macro_real_income", "macro_real_gdp", "macro_unemployment",
    "macro_cpi", "macro_consumer_sentiment",
]

MODEL_COLORS = {
    "Naive (zero)"    : NEUTRAL,
    "Ridge Regression": PALETTE[0],
    "Random Forest"   : PALETTE[1],
    "SVM"             : PALETTE[4],
    "XGBoost"         : PALETTE[2],
}

print("Imports OK")

## 2. Load Data & Compute Target

We load `basetable.csv` and keep only the 27 financial columns.

**Target derivation:**  
`target(t) = polymarket_trump_prob(t) − polymarket_trump_prob(t−1)`

After `dropna()`, the dataset shrinks dramatically to ~32 rows because monthly macro indicators only have values on release days. This is an inherent limitation of including macro data.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=["date"])
df_raw = df_raw.sort_values("date").reset_index(drop=True)

missing = [c for c in FIN_COLS if c not in df_raw.columns]
assert not missing, f"Missing financial columns: {missing} — check basetable.csv"

df = df_raw[["date", "polymarket_trump_prob"] + FIN_COLS].copy()
df["target"] = df["polymarket_trump_prob"] - df["polymarket_trump_prob"].shift(1)
df = df.dropna().reset_index(drop=True)

print(f"Rows          : {len(df)}")
print(f"Date range    : {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Features ({len(FIN_COLS)}): {len(FIN_COLS)} financial columns")
print(f"\nTarget (daily delta prob) stats:")
print(df["target"].describe().round(5))

if len(df) < 50:
    print(f"\n[WARNING] Only {len(df)} rows available. Model estimates will have very high variance.")
    print("          This is caused by monthly/quarterly macro indicators limiting the dataset.")

## 3. Train/Val/Test Split

`get_test_split` carves off the **last 14 rows** as a completely held-out test set. With only ~32 total rows, this leaves only ~18 rows for training and validation — a fundamental constraint of this dataset.

In [ ]:
tv_idx, test_idx = get_test_split(df, test_days=TEST_DAYS)

df_tv   = df.iloc[tv_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

X_tv   = df_tv[FIN_COLS].values
y_tv   = df_tv["target"].values
X_test = df_test[FIN_COLS].values
y_test = df_test["target"].values

print(f"Train/val : {len(df_tv):>4} rows  ({df_tv['date'].min().date()} -> {df_tv['date'].max().date()})")
print(f"Test      : {len(df_test):>4} rows  ({df_test['date'].min().date()} -> {df_test['date'].max().date()})")

## 4. CV Folds

**3 expanding-window folds** with a 1-day gap. With ~18 TV rows and 3 splits, each fold has only 4–5 validation rows and 5–13 training rows — estimates are noisy but the walk-forward structure is correct: no future data is ever used to train or evaluate.

In [ ]:
folds = get_cv_folds(df_tv, n_splits=N_SPLITS, gap=GAP, test_days=None)
print_fold_summary(df_tv, folds)

print("\nLeakage validation:")
for i, (tr, va) in enumerate(folds, 1):
    validate_no_leakage(tr, va, df_tv, gap=GAP)
    print(f"  Fold {i}: OK")

## 5. Helper Functions

All helpers are imported from `Functions.evaluation_metrics`. Scalers are always fitted exclusively on training data.

## 6. Model 1 — Ridge Regression

With 27 features and only 5–13 training rows per fold, plain OLS is severely underdetermined (n_train << p): infinitely many solutions exist and coefficients are numerically undefined without regularisation. Ridge resolves this by selecting the minimum-norm solution via the L2 penalty `α‖β‖²`.

**Tuned hyperparameter:** `alpha` — higher alpha = stronger shrinkage toward zero; needed here to prevent wildly large coefficients on the 5-row fold.  
**Scaling applied** — mandatory for Ridge to penalise all features equally.

In [ ]:
def make_ridge(alpha):
    return Ridge(alpha=alpha)

# Wider grid: with n_train=5, optimal alpha may be very large
ridge_param_grid = {"alpha": [0.1, 1.0, 10.0, 100.0, 500.0, 1000.0, 5000.0]}

print("=== Ridge Regression — Hyperparameter Tuning ===")
ridge_best, ridge_tune_df = tune_hyperparams(
    make_ridge, ridge_param_grid, folds, X_tv, y_tv, scale=True
)
print(f"  Best params : {ridge_best}")
print(f"\n  All configurations (sorted by CV MAE):")
print(ridge_tune_df.to_string(index=False))

ridge_factory = lambda: make_ridge(**ridge_best)
print("\n=== Ridge Regression — CV (best params) ===")
ridge_cv = cv_evaluate(ridge_factory, folds, X_tv, y_tv, scale=True)
ridge_cv.round(4)

In [ ]:
ridge_model, ridge_pred, ridge_test = final_eval(
    ridge_factory, X_tv, y_tv, X_test, y_test, scale=True
)
print(f"Ridge Regression — Test set  (alpha={ridge_best['alpha']}):")
for k, v in ridge_test.items():
    print(f"  {k}: {v:.4f}")

## 7. Model 2 — Random Forest Regressor

Random Forest is robust to small samples: bootstrap sampling introduces diversity even from small training sets, and leaf regularisation (`min_samples_leaf`) prevents individual trees from fitting single points. We enforce shallow trees (`max_depth ≤ 4`) to avoid memorising the 5–13 training rows.

**Fixed:** `n_estimators=200`, `random_state=42`.  
**Tuned:** `max_depth`, `min_samples_leaf` — both critical for preventing overfitting on tiny folds.  
**No scaling required.**

In [ ]:
def make_rf(max_depth, min_samples_leaf):
    return RandomForestRegressor(
        n_estimators=200,
        max_depth=int(max_depth),
        min_samples_leaf=int(min_samples_leaf),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

rf_param_grid = {
    "max_depth"       : [2, 3, 4],
    "min_samples_leaf": [2, 3, 5],
}

print("=== Random Forest — Hyperparameter Tuning ===")
rf_best, rf_tune_df = tune_hyperparams(
    make_rf, rf_param_grid, folds, X_tv, y_tv, scale=False
)
print(f"  Best params : {rf_best}")
print(f"\n  All configurations (sorted by CV MAE):")
print(rf_tune_df.to_string(index=False))

rf_factory = lambda: make_rf(**rf_best)
print("\n=== Random Forest — CV (best params) ===")
rf_cv = cv_evaluate(rf_factory, folds, X_tv, y_tv, scale=False)
rf_cv.round(4)

In [ ]:
rf_model, rf_pred, rf_test = final_eval(
    rf_factory, X_tv, y_tv, X_test, y_test, scale=False
)
print("Random Forest — Test set:")
for k, v in rf_test.items():
    print(f"  {k}: {v:.4f}")

fi = pd.Series(rf_model.feature_importances_, index=FIN_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
bars = ax.barh(fi.index, fi.values, color=MODEL_COLORS["Random Forest"], alpha=0.85, height=0.6)
for bar, v in zip(bars, fi.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{v:.3f}", va="center", ha="left", fontsize=8, color=TEXT_MUTED)
ax.set_xlabel("Mean Decrease in Impurity", color=TEXT_PRIMARY)
ax.set_title(f"Random Forest — Feature Importances  |  best: {rf_best}",
             color=TEXT_PRIMARY, fontweight="bold", fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
ax.tick_params(colors=TEXT_MUTED)
ax.set_yticklabels(fi.index, color=TEXT_PRIMARY, fontsize=8)
ax.grid(axis="x", color=GRID_COLOR, linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 8. Model 3 — Support Vector Regression (tuned kernel)

SVR's dual formulation depends on the number of support vectors (bounded by n_train), not the number of features — it is therefore better suited to the tiny training sets here than primal methods. The linear kernel effectively applies Ridge-like regularisation within SVR's margin framework.

**Fixed:** `gamma='scale'`.  
**Tuned:** `kernel`, `C`, `epsilon`.  
**Scaling required.**

In [ ]:
def make_svr(kernel, C, epsilon):
    return SVR(kernel=kernel, C=C, epsilon=epsilon, gamma="scale")

svr_param_grid = {
    "kernel" : ["linear", "rbf"],
    "C"      : [0.01, 0.1, 1.0, 10.0],
    "epsilon": [0.001, 0.01, 0.05],
}

print("=== SVM — Hyperparameter Tuning ===")
svr_best, svr_tune_df = tune_hyperparams(
    make_svr, svr_param_grid, folds, X_tv, y_tv, scale=True
)
print(f"  Best params : {svr_best}")
print(f"\n  All configurations (sorted by CV MAE):")
print(svr_tune_df.to_string(index=False))

svr_factory = lambda: make_svr(**svr_best)
print("\n=== SVM — CV (best params) ===")
svm_cv = cv_evaluate(svr_factory, folds, X_tv, y_tv, scale=True)
svm_cv.round(4)

In [ ]:
svm_model, svm_pred, svm_test = final_eval(
    svr_factory, X_tv, y_tv, X_test, y_test, scale=True
)
print(f"SVM — Test set  (kernel={svr_best['kernel']}, C={svr_best['C']}, ε={svr_best['epsilon']}):")
for k, v in svm_test.items():
    print(f"  {k}: {v:.4f}")

## 9. Model 4 — XGBoost Regressor

XGBoost sequentially adds shallow trees. With 5–13 training rows, we constrain the model heavily:
- `max_depth ≤ 3` — prevents trees from perfectly fitting individual training points
- `colsample_bytree=0.5` — uses only 50 % of features per tree (≈ 13 of 27), reducing variance  
- `subsample=0.8` — stochastic boosting for additional regularisation

**Fixed:** `subsample=0.8`, `colsample_bytree=0.5`, `objective='reg:squarederror'`, `random_state=42`.  
**Tuned:** `max_depth`, `learning_rate`, `n_estimators`.  
**No scaling required.**

In [ ]:
def make_xgb(max_depth, learning_rate, n_estimators):
    return XGBRegressor(
        max_depth=int(max_depth),
        learning_rate=learning_rate,
        n_estimators=int(n_estimators),
        subsample=0.8,
        colsample_bytree=0.5,   # reduced from 0.8 because n_train << n_features
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        verbosity=0,
    )

xgb_param_grid = {
    "max_depth"    : [2, 3],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators" : [50, 100],
}

print("=== XGBoost — Hyperparameter Tuning ===")
xgb_best, xgb_tune_df = tune_hyperparams(
    make_xgb, xgb_param_grid, folds, X_tv, y_tv, scale=False
)
print(f"  Best params : {xgb_best}")
print(f"\n  All configurations (sorted by CV MAE):")
print(xgb_tune_df.to_string(index=False))

xgb_factory = lambda: make_xgb(**xgb_best)
print("\n=== XGBoost — CV (best params) ===")
xgb_cv = cv_evaluate(xgb_factory, folds, X_tv, y_tv, scale=False)
xgb_cv.round(4)

In [ ]:
xgb_model, xgb_pred, xgb_test = final_eval(
    xgb_factory, X_tv, y_tv, X_test, y_test, scale=False
)
print("XGBoost — Test set:")
for k, v in xgb_test.items():
    print(f"  {k}: {v:.4f}")

fi_xgb = pd.Series(xgb_model.feature_importances_, index=FIN_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
bars = ax.barh(fi_xgb.index, fi_xgb.values, color=MODEL_COLORS["XGBoost"], alpha=0.85, height=0.6)
for bar, v in zip(bars, fi_xgb.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{v:.3f}", va="center", ha="left", fontsize=8, color=TEXT_MUTED)
ax.set_xlabel("Feature Importance (gain)", color=TEXT_PRIMARY)
ax.set_title(f"XGBoost — Feature Importances (gain)  |  best: {xgb_best}",
             color=TEXT_PRIMARY, fontweight="bold", fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
ax.tick_params(colors=TEXT_MUTED)
ax.set_yticklabels(fi_xgb.index, color=TEXT_PRIMARY, fontsize=8)
ax.grid(axis="x", color=GRID_COLOR, linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 10. Naive Baseline — Always Predict Zero

The naive model always predicts `Δ prob = 0`. With only 4-row validation folds, even the naive MAE estimate is very noisy, but it provides the correct reference point for R².

In [ ]:
print("=== Naive (zero) — CV ===")
naive_records = []
for i, (train_idx, val_idx) in enumerate(folds, 1):
    y_val  = y_tv[val_idx]
    y_zero = np.zeros_like(y_val)
    m = {"Fold": i, **compute_metrics(y_val, y_zero)}
    naive_records.append(m)
    print(f"  Fold {i}: MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  "
          f"DA={m['Dir. Accuracy']:.3f}  R2={m['R2']:.4f}")

naive_agg  = pd.DataFrame(naive_records).set_index("Fold")
naive_mean = naive_agg.mean().rename("Mean")
naive_std  = naive_agg.std().rename("Std")
naive_cv   = pd.concat([naive_agg, naive_mean.to_frame().T, naive_std.to_frame().T])
print(f"  -- Mean --  MAE={naive_mean['MAE']:.4f}  RMSE={naive_mean['RMSE']:.4f}  "
      f"DA={naive_mean['Dir. Accuracy']:.3f}  R2={naive_mean['R2']:.4f}")

naive_pred = np.zeros(len(y_test))
naive_test = compute_metrics(y_test, naive_pred)
print("\nNaive (zero) — Test set:")
for k, v in naive_test.items():
    print(f"  {k}: {v:.4f}")

naive_cv.round(4)

## 11. Model Comparison

CV performance (mean over 3 folds) and test-set performance for all five models.

**Caveat:** with only 4-row validation folds and 5–13 training rows, CV estimates have very high variance. The test-set results (14 rows) are more meaningful but still limited. Interpret all financial model results with caution.

In [ ]:
cv_summary = pd.DataFrame({
    "Naive (zero)"    : naive_cv.loc["Mean"],
    "Ridge Regression": ridge_cv.loc["Mean"],
    "Random Forest"   : rf_cv.loc["Mean"],
    "SVM"             : svm_cv.loc["Mean"],
    "XGBoost"         : xgb_cv.loc["Mean"],
}).T.round(4)

best_params_col = {
    "Naive (zero)"    : "— (always 0)",
    "Ridge Regression": f"alpha={ridge_best['alpha']}",
    "Random Forest"   : f"d={rf_best['max_depth']}, leaf={rf_best['min_samples_leaf']}",
    "SVM"             : f"k={svr_best['kernel']}, C={svr_best['C']}, ε={svr_best['epsilon']}",
    "XGBoost"         : f"d={xgb_best['max_depth']}, lr={xgb_best['learning_rate']}, n={int(xgb_best['n_estimators'])}",
}
cv_summary.insert(0, "Best params", pd.Series(best_params_col))

print("CV performance (mean across 3 walk-forward folds):")
print("[Note: 4-row validation folds — estimates are noisy]")
display(cv_summary)

In [ ]:
test_summary = pd.DataFrame({
    "Naive (zero)"    : naive_test,
    "Ridge Regression": ridge_test,
    "Random Forest"   : rf_test,
    "SVM"             : svm_test,
    "XGBoost"         : xgb_test,
}).T.round(4)

print("Test set performance:")
display(test_summary)

In [ ]:
test_dates = df_test["date"].values
preds_list = [
    ("Naive (zero)",    naive_pred),
    ("Ridge Regression", ridge_pred),
    ("Random Forest",   rf_pred),
    ("SVM",             svm_pred),
    ("XGBoost",         xgb_pred),
]

fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True)
fig.patch.set_facecolor(BG_DARK)

for ax, (label, pred) in zip(axes, preds_list):
    ax.set_facecolor(BG_PANEL)
    ax.plot(test_dates, y_test, label="Actual",    color=TEXT_PRIMARY, linewidth=2)
    ax.plot(test_dates, pred,   label="Predicted", color=MODEL_COLORS[label],
            linewidth=1.8, linestyle="--")
    ax.axhline(0, color=TEXT_MUTED, linewidth=0.9, linestyle=":")
    da  = directional_accuracy(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    ax.set_title(f"{label}  |  DA={da:.2f}  MAE={mae:.4f}",
                 color=MODEL_COLORS[label], fontweight="bold", fontsize=10)
    ax.set_ylabel("Delta prob", color=TEXT_PRIMARY, fontsize=9)
    ax.legend(loc="upper right", facecolor=BG_PANEL, edgecolor=SPINE_COLOR,
              labelcolor=TEXT_PRIMARY, fontsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.7)
    ax.set_axisbelow(True)

axes[-1].set_xlabel("Date", color=TEXT_PRIMARY)
plt.xticks(rotation=25, ha="right")
fig.suptitle("Test-set predictions vs actuals\nDaily delta Polymarket Trump probability — Financial features only",
             color=TEXT_PRIMARY, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
metrics_plot = ["MAE", "RMSE", "Dir. Accuracy", "R2"]
model_names  = list(MODEL_COLORS.keys())
colors_list  = list(MODEL_COLORS.values())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.patch.set_facecolor(BG_DARK)

for ax, metric in zip(axes, metrics_plot):
    ax.set_facecolor(BG_PANEL)
    vals = [test_summary.loc[m, metric] for m in model_names]
    bars = ax.bar(range(len(model_names)), vals, color=colors_list, alpha=0.85, width=0.6)

    if metric == "Dir. Accuracy":
        ax.axhline(0.5, color=TEXT_MUTED, linestyle="--", linewidth=1.2, label="Coin flip (0.5)")
    if metric == "R2":
        ax.axhline(0, color=TEXT_MUTED, linestyle="--", linewidth=1.2, label="Naive / mean pred.")

    for bar, val in zip(bars, vals):
        offset = abs(val) * 0.03 + 0.001
        va     = "bottom" if val >= 0 else "top"
        y_pos  = bar.get_height() + offset if val >= 0 else bar.get_height() - offset
        ax.text(bar.get_x() + bar.get_width()/2, y_pos,
                f"{val:.3f}", ha="center", va=va, fontsize=7.5, color=TEXT_MUTED)

    ax.set_title(metric, color=TEXT_PRIMARY, fontweight="bold")
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in model_names],
                       fontsize=7.5, color=TEXT_MUTED)
    for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.7)
    ax.set_axisbelow(True)
    if metric in ("Dir. Accuracy", "R2"):
        ax.legend(facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, fontsize=8)

fig.suptitle("Test-set metric comparison — Financial-only baseline",
             color=TEXT_PRIMARY, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()